# DataFrame Classification Validation (FlashSequential)

Validate a `FlashSequential` classifier against an existing validation subset.
Run this notebook below a training notebook, or set `model_path` and `validation_csv` to work standalone.

In [ ]:
# Leave these as None when attaching this notebook below a training notebook.
model_path = None  # optional path to a saved Keras model
validation_csv = None  # optional CSV containing features and the target
target_column = "target"
separator = ","

## 1. Use the model and validation data

In [ ]:
import numpy as np
import pandas as pd
from flashkeras import FlashSequential

if model_path is not None:
    model = FlashSequential("classification")
    model.loadModel(model_path)
elif "model" not in globals():
    raise NameError("Define `model` in a previous cell or set `model_path`.")

if validation_csv is not None:
    validation_df = pd.read_csv(validation_csv, sep=separator)
    if target_column not in validation_df.columns:
        raise KeyError(f"Target column not found: {target_column}")
    x_val = validation_df.drop(columns=[target_column])
    y_val = validation_df[target_column]
elif not {"x_val", "y_val"}.issubset(globals()):
    raise NameError("Define `x_val` and `y_val` in a previous cell or set `validation_csv`.")

print(f"Validation samples: {len(y_val)}")
print(f"Features: {x_val.shape[1]}")

## 2. Evaluate metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

x_eval = x_val.to_numpy() if isinstance(x_val, pd.DataFrame) else x_val
y_eval = y_val.to_numpy() if isinstance(y_val, (pd.Series, pd.DataFrame)) else y_val
evaluation = model.model.evaluate(x_eval, y_eval, return_dict=True, verbose=0)
probabilities = model.predict(x_eval, verbose=0)

if probabilities.ndim == 2 and probabilities.shape[1] == 1:
    predicted_classes = (probabilities[:, 0] >= 0.5).astype(int)
else:
    predicted_classes = np.argmax(probabilities, axis=1)

true_classes = y_eval
if true_classes.ndim > 1 and true_classes.shape[1] > 1:
    true_classes = np.argmax(true_classes, axis=1)
labels = np.unique(np.concatenate([true_classes, predicted_classes]))

print("Keras evaluation:")
for metric_name, value in evaluation.items():
    print(f"{metric_name}: {value:.4f}")
print("\nClassification report:")
print(classification_report(true_classes, predicted_classes, labels=labels, zero_division=0))

ConfusionMatrixDisplay(
    confusion_matrix(true_classes, predicted_classes, labels=labels),
    display_labels=labels,
).plot()
plt.tight_layout()